**Author:** Hui Fang\
**Purpose:** ST 554 Final Project\
Date: 4/20/2026

# Introction

This project demonstrates the end-to-end use of `PySpark` for both machine learning and real-time data processing. The data is adapted from the [UCI machine learning repository](https://archive.ics.uci.edu/dataset/849/power+consumption+of+tetouan+city), which examines how power consumption across different zones of Tetouan city relates to various factors such as time of day, temperature, and humidity.\
The project consistsof two components. The first component focuses on building an **Elastic Net regression model** using `Spark MLlib`, incorporating SQL-based feature engineering, one-hot encoding, PCA, and cross-validated hyperparameter tuning. The second component extends this work into a streaming context: a `Python` script generates incoming CSV files, `Spark` monitors the folder as a stream, applies the fitted model to produce predictions and residuals in real time, and outputs the results to the console.\
 Together, these components highlight Spark's ability to unify batch modeling and streaming inference within a single, conherent workflow.

# Goals
- create or use an already created github repo to document your work
    - You must commit often! At least 5 substantial commits must exist on this project. This allows us to see how your work develops over time. If this is not done, substantial credit will be lost.
- write a Jupyter notebook that fits a machine learning model using **pyspark's MLlib** module
- in that same notebook you'll write code to read in a stream of data (we'll produce that data ourselves using a **.py** file)
    - This **.py** file should also be included in the repo.
- we'll use the model to do predictions on the stream and write those out to the console!

Both the **.ipynb** file and the **.py** file should exist in your repo!

The data is modified from the UCI machine learning repository. The study was about relating power consumption from different zones of Tetouan city to various factors like time of day, temperature, and humidity.
- You'll have a chunk of the (modified) data to build your model on.
- You will then 'stream data' to a folder in the form of CSV files. You'll be monitoring this folder. When data comes in you'll use your fitted model to predict on the incoming data!

# Part I: Fitting Model
**Create a Jupyter notebook for the modeling fitting part and the Streaming part below.**
- The file power_ml_data.csv is available at the URL: https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv
- You should read this data into a standard pandas data frame using the **pd.read_csv()** function.
- Convert this to a spark data frame
- We are going to treat the **Power_Zone_3** variable as our response variable.
- We can use all of the other variables as predictors. (Imagine we know that the **Power_Zone_3** reading is going to go offline in the future and we need to be able to predict that value appropriately.)

We want to fit an elastic net model using CV (no training/test split, just using CV on the data we've read in) with the steps below.\
The transformations below should each use an **MLlib** function that can be put into a pipeline
- The Hour column is likely not stored as a DoubleType. If it is not, use an SQL transformer to cast the
variable as a DoubleType
- Binarize the Hour column based on the column being less than 6.5 or not (night vs day essentially)
- One-hot encode the Month column
- Run a `PCA fit` on the **Temperature**, **Humidity**, **Wind_Speed**, **general_Diffuse_Flows**, and **Diffuse_Flows** columns.
    - To do this, I first used a VectorAssembler() call to place these variables in a column together for use with the PCA() estimator.
    - Once fitted, then you'll have a PCA transformer we'll use in our pipeline.
    - We'll use two PCs in our transformation.
- Rename your response variable as label
- Use VectorAssembler() to put your predictors into a features. Use the
    -  two fitted PCA features
    - binary **Hour** variable
    - **Power_Zone_1**
    - **Power_Zone_2**
    - **Month** indicator variables
- This ends the pipeline of transformations!
- Now you'll then use the **CrossValidator()** function and the **LinearRegression()** function to fit an elastic net model.
    - You should do the following grid for the regParam and elasticNetParam: All combinations of\
    ***regParam:** 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1\
    ***elasticNetParam:** 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1
- Now fit the model using 5-fold CV with **rmse** as your criterion!
- Report the optimal values chosen for the tuning parameters
- Report the CV error
- Report the training set RMSE (as done in the notes) by using your fitted model as a transformer and
evaluating on the entire training set
- Take the outputted transformations from the model (the predictions) and create a **residual** column (**label - prediction**). The **.withColumn()** method is handy here. Print out a data frame with these **residuals**, the **label** column, and the **predictions**.

First let's import the modules needed and create a Spark session.

In [2]:
# import modules needed
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml.feature import SQLTransformer, Binarizer, OneHotEncoder, VectorAssembler, PCA
from pyspark.ml import Pipeline

# Create a Spark session
spark = SparkSession.builder.getOrCreate()

# prevent producing long list of warnings
spark.sparkContext.setLogLevel("ERROR")

## 1. Load data & convert to Spark DataFrame

In [3]:
# read in data
power_pd = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv")
power_pd.head()
# convert to Spark Dataframe
power_sdf = spark.createDataFrame(power_pd)
# print out the data frame schema
power_sdf.printSchema()
# check the spark DataFrame
power_sdf.show(10)

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
|      5.853|    76.9|     0.081|       

## 2. Variable transformation

- **Cast `Hour` to double:** The above printed schema indicates that `Hour` is stored as a long. To ensure compatibility with downstream `MLlib` transformations, I cast `Hour` to `DoubleType` using an `SQLTransformer` within the pipeline.
- **Binarize the `Hour` and One-hot encode `Month`:** Next, I create a binary indicator for the `Hour` variable, where values less than 6.5 represent nighttime and values greater than or equal to 6.5 represent daytime. Since `Month` is already stored as a numeric type, we can directly apply OneHotEncoder without any additional casting.

In [4]:
# cast Hour vaiable to double
hour_cast_sql = SQLTransformer(statement = "SELECT *, CAST(Hour AS DOUBLE) AS Hour_double FROM __THIS__")

# use Binarizer on Hour_double
hour_bin = Binarizer(
    inputCol = "Hour_double",
    outputCol = "Hour_binary",
    threshold = 6.5)

# One-hot encode the month variable
month_encoder = OneHotEncoder(
    inputCols = ["Month"],
    outputCols = ["Month_ohe"])

## 3. Fit a PCA Model

To fit a PCA (Principal Component Analysis), which reduces several correlated variables into a smaller set of summary components, I first assemble `Temperature`, `Humidity`, `Wind_Speed`, `General_Diffuse_Flows`, and `Diffuse_Flows` into a single feature vector using `VectorAssembler()`. After  fitting the PCA model, Spark produces a PCA transformer that becomes part of the pipeline. In this project, I retain two principal components (PCs).

In [5]:
# assemble variables into a single vector
pca_assembler = VectorAssembler(
    inputCols = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows","Diffuse_Flows"],
    outputCol = "pca_input")

# fit a PCA model
pca = PCA(
    k = 2,
    inputCol = "pca_input",
    outputCol = "pca_features")

## 4. Assemble the final `features` vector

Next, I add an `SQLTransformer` to rename `Power_Zone_3` as `label`. I then use a final `VectorAssembler` to combine predictors `Hour_binary`, `Power_Zone_1`, `Power_Zone_2`, the one-hot encoded `Month_ohe`, and the two PCA compoents (`pca_features`) into a gingle `features` column for model fitting.

In [6]:
# rename sesponse `Power_Zone_3` to `label`
label_sql = SQLTransformer(
    statement = """
        SELECT *, Power_Zone_3 AS label
        FROM __THIS__
    """
)

# combine preditors into `features`
features_assembler = VectorAssembler(
    inputCols = [
        "Hour_binary",
        "Power_Zone_1",
        "Power_Zone_2",
        "Month_ohe",
        "pca_features"
    ],
    outputCol = "features")

## 5. Build the transformation pipeline

Now, I chain all transformation stages together into a single Spark MLlib pipeline, ensuring that each transformation is applied in the correct order before model fitting.

In [7]:
# build the transformation pipeline
transform_stages = [
    hour_cast_sql,
    hour_bin,
    month_encoder,
    label_sql,
    pca_assembler,
    pca,
    features_assembler]

transform_pipeline = Pipeline(stages = transform_stages)

# fit the transformation pipeline
transform_model = transform_pipeline.fit(power_sdf)

# Apply the fitted transformation pipeline to the original DataFrame
train_transformed = transform_model.transform(power_sdf)

# Display the first five rows of the label and features columns for checking
train_transformed.select("label", "features").show(5, truncate = False)

+-----------+---------------------------------------------------------------------------------------+
|label      |features                                                                               |
+-----------+---------------------------------------------------------------------------------------+
|20240.96386|(17,[1,2,4,15,16],[34055.6962,16128.87538,1.0,1.794404863656954,-0.7412746447404285])  |
|20131.08434|(17,[1,2,4,15,16],[29814.68354,19375.07599,1.0,1.8060408300982311,-0.7108534239558307])|
|19668.43373|(17,[1,2,4,15,16],[29128.10127,19006.68693,1.0,1.8102297630563902,-0.7283113191158763])|
|18899.27711|(17,[1,2,4,15,16],[28228.86076,18361.09422,1.0,1.7986676517408842,-0.7220913032199767])|
|18442.40964|(17,[1,2,4,15,16],[27335.6962,17872.34043,1.0,1.863287201637971,-0.7323046647776382])  |
+-----------+---------------------------------------------------------------------------------------+
only showing top 5 rows


The output confirms that the transformation pipeline executed correctly: the response variable has been renamed to `label`, and all predictors have been successfully assembled into the unified `features` vector required for model fitting.

## 6. Fit the Elastic Net model

Once the transformation pipeline has produced the final `label` and `features` columns, I proceed to fit the `Elastic Net` regression model. I begin by importing the required modules.

In [8]:
# import modules needed
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder
from pyspark.ml.tuning import CrossValidator

After completing the transformation pipeline, we proceed to fit the `Elastic Net` regression model. This involves instantiating the LinearRegression estimator, constructing the hyperparameter grid for both regularization strength and Elastic Net mixing, defining the RMSE evaluator, configuring the 5-fold CrossValidator, and finally fitting the cross-validated model on the transformed dataset.

In [9]:
# create a regression model instance
lr = LinearRegression(
    featuresCol = "features", 
    labelCol = "label",
    maxIter=5000,
    solver="auto")

# build the parameter grid
paramGrid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1])
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1])
    .build()
)

# define the evaluator
evaluator = RegressionEvaluator(
    predictionCol = "prediction",
    labelCol = "label",
    metricName = "rmse")

# set up the CrossValidator
cv = CrossValidator(
    estimator = lr,
    estimatorParamMaps = paramGrid,
    evaluator = evaluator,
    numFolds = 5,                   # set 5-fold CV
    parallelism = 8,                # run up to 8 CV tasks simultaneously
    seed = 42                       # set seed for reproducibility
)

Now, I fit the cross-validated `Elastic Net` model on the transformed dataset. This step trains all 121 hyperparameter combinations, evaluates each using 5-fold cross-validation, and automatically selects the model with the lowest RMSE. Because the grid is large, it may take several mintues to fit the CV model.

In [10]:
# fit the CV model on transformed data
cv_model = cv.fit(train_transformed)

26/04/30 09:14:00 ERROR LBFGS: Failure! Resetting history: breeze.optimize.FirstOrderException: Line search zoom failed
26/04/30 09:14:00 ERROR LBFGS: Failure again! Giving up and returning. Maybe the objective is just poorly behaved?
26/04/30 09:14:01 ERROR LBFGS: Failure! Resetting history: breeze.optimize.FirstOrderException: Line search zoom failed
26/04/30 09:14:01 ERROR LBFGS: Failure again! Giving up and returning. Maybe the objective is just poorly behaved?


After fitting the model, I extract the best hyperparameters to inspect the best `Elastic Net` model and report the optimal tuning parameters.

In [11]:
# extract the best model
best_model = cv_model.bestModel

# extract the best tuning parameters
best_reg = best_model._java_obj.getRegParam()
best_alpha = best_model._java_obj.getElasticNetParam()

# print out the best tunning parameters
print("Best regParam (λ):", best_reg)
print("Best elasticNetParam (α):", best_alpha)

# report the CV error (RMSE)
best_cv_rmse = min(cv_model.avgMetrics)
print("Best CV RMSE:", best_cv_rmse)

Best regParam (λ): 0.05
Best elasticNetParam (α): 0.1
Best CV RMSE: 2147.589787258122


The results show that the optimal tuning parameters selected by cross-validation are λ = 0.1 and α = 0.25, with the corresponding minimum RMSE of 2147.59.\
Using these best-performing hyperparameters, I can now generate predictions with the fitted Elastic Net model.

In [12]:
# make prediction
train_predictions = best_model.transform(train_transformed)

# extract and print out training RMSE
train_rmse = evaluator.evaluate(train_predictions)
print("Training RMSE:", train_rmse)

Training RMSE: 2147.097316929394


The trarining set RMSE is 2147.10, which is very close to but slightly lower than the cross-validated RMSE. The small difference indicates that the selected `Elastic Net` model generalizes well and does not exhibit overfitting.

Next, I compute the residuals for the fitted model and present a DataFrame that includes the true `label`, the model's prediction, and the residual (defined as `label - prediction`).

In [13]:
# import module needed
from pyspark.sql.functions import col

# create residual data frame
residuals_df = (
    train_predictions
    .withColumn("residual", col("label") - col("prediction"))
    .select("label", "prediction", "residual")
)
# show the first 10 rows of the results
residuals_df.show(10, truncate = False)

+-----------+------------------+------------------+
|label      |prediction        |residual          |
+-----------+------------------+------------------+
|20240.96386|20878.850660788943|-637.8868007889432|
|20131.08434|18660.227266543912|1470.857073456089 |
|19668.43373|18204.75215311442 |1463.6815768855813|
|18899.27711|17590.648498338982|1308.6286116610172|
|18442.40964|16997.301986456696|1445.1076535433058|
|18130.12048|16517.686723494102|1612.4337565058995|
|17945.06024|16093.246141053285|1851.8140989467138|
|17459.27711|15722.695360253707|1736.5817497462922|
|17025.54217|15271.043828662609|1754.498341337392 |
|16794.21687|14938.34876466541 |1855.86810533459  |
+-----------+------------------+------------------+
only showing top 10 rows


# Part II: Streaming

For Part II, I construct a streaming workflow by reading .csv files as they arrive in a designated input directory. I generate these files by randomly sampling rows from the full dataset and writing them out sequentially using a python script, thereby emulating a real-time data stream. The original dataset can be accessed [here](https://www4.stat.ncsu.edu/~online/datasets/power_streaming_data.csv).

## 1. Load data and obtain schema
- We're going to read in a stream in the form of .csv files. Create a folder where you will be sending
your .csv files.
- Setup the schema for the stream (you can use the schema from the original data as we did in hw 10)
- Set up the readStream. Be sure to add header = True as you'll likely be outputting files with a header and we don't need to read that in.

I load the full dataset once in batch mode to inspect the structure and extract a schema. Spark requires an explicit schema for Structured Streaming because it cannot infer column types from streaming files. This schema is then applied to all incoming CSV files generated by the Python script.

In [14]:
# import modules needed
from pyspark.sql import SparkSession
from pyspark.ml.feature import SQLTransformer, Binarizer, OneHotEncoder, VectorAssembler, PCA

# Create a Spark session
spark = SparkSession.builder.getOrCreate()

# suppress verbose Spark info and warn messages
spark.sparkContext.setLogLevel("ERROR")

## 2. Reading the stream
First, I want to ensure that the streaming input directory exists and then extract the schema from the batch DataFrame. I create the streaming input directory (if needed), extract the schema from the batch DataFrame, and configure Spark to continuously monitor the folder for new CSV files.

In [15]:
# extract the schema from the static Spark DataFrame
schema = power_sdf.schema

# turn the folder into a live data source
stream_df = (
    spark.readStream
        .schema(schema)            # apply the predefined schema
        .option("header", True)    # ensure incoming files include a header row
        .csv("stream_input"))      # folder where streaming files will appear

## 3. Transform and Aggregation 
- Now, we'll do two separate things on the stream and join them together:
    - With your stream, use your model transformer to obtain predictions from the incoming data. On the resulting predictions also create a **residual** column as noted in the previous section (return only the **label**, **prediction** and **residual** columns from this part)
    - We can use our stream more than once! With another transformation on the (original) stream, modify the response variable to be called **label**.
    - Now join your above transform with this stream based on the **label** variable which should be common to both!
    - Note 1: This is a little silly, but I want you to join two transformations of the stream and I don't want things to get too crazy
    - Note 2: Each data frame is created from the same stream of data! You don't need two streams, you can use the same stream and just do two separate transformations on it, combining it with a **.join()** method from one of the SQL style data frames you are dealing with (as we discussed in the notes)


In this step, I process the incoming data stream to prepare it for model prediction and then generate predictions and residuals.
First, I rename the original response variable in the raw `stream_df` to `label` (stored as `label_df`), which will be used for joining with predictions later.
Next, I apply the *entire fitted transformation pipeline* from Part-I to the `stream_df` (creating `stream_transformed`). This ensures all feature engineering steps are consistently applied, preparing the data with the same `features` vector used during training.\
Finally, I apply the trained Elastic Net regression `cv_model` to the `stream_transformed` data to obtain predictions. I then compute residuals (the difference between the observed `label` and the `prediction`) and retain only the `label`, `prediction`, and `residual` columns in the `pred_df` for output.

In [16]:
# import modules needed
import pyspark
from pyspark.sql.functions import col

# build a streaming transformer
stream_transformer = transform_model

# rename the response variable to 'label'
label_df = stream_df.withColumnRenamed("Power_Zone_3", "label")

# apply the streaming transformer
stream_transformed = stream_transformer.transform(stream_df)

# apply the trained Elastic Net regression model to obtain predictions on the streaming data
pred_df = cv_model.transform(stream_transformed)

# compute residuals for each incoming record
pred_df = (pred_df
           .withColumn("residual", col("label") - col("prediction"))
           .select("label", "prediction", "residual") # only keep the target columns
        )

# join the prediction DataFrame with the renamed raw stream on the 'label'
joined_df = pred_df.join(label_df, on = "label")

# check the schema of joined data frame
joined_df.printSchema()

root
 |-- label: double (nullable = true)
 |-- prediction: double (nullable = false)
 |-- residual: double (nullable = true)
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



The output confirms that the schema of the joined DataFrame includes the label, prediction, residual, and all expected predictor variables.

## 4. Start the streaming query 
- Now write your stream to the console using the append output mode.
- Start the query!

You should have the file we'll use for streaming data downloaded and in a place you can locate. Create a .py file that reads that into a pandas (regular) data frame and does the following.
- Writes a loop (say 20 iterations) to:
    - Randomly sample five rows and output those to a .csv file in the folder you are watching with your stream.
    - Be sure not to write out the indices. You can leave the column names as long as you handle that on your stream appropriately
    - Pause for 10 seconds in between outputting of data sets
    - Submit this loop in a python console.
- While the loop runs, you should see output in your notebook!




Finally, I configure the streaming sink. Using append mode, the console sink prints each processed micro-batch to the console as new files arrive in the input directory while the Python script is running.

In [17]:
# start the query and write output to the console in append mode
query = (
    joined_df.writeStream
             .outputMode("append")                 # append new rows as they arrive
             .format("console")                    # print results to the console
             .option("truncate", False)            # print all columns
             .option("numRows", 5)                 # print only the first 5 rows
             .trigger(processingTime = "1 second") # add a trigger
             .start()                              # start the streaming query
)

-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual           |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|15886.45161|16288.358500250979|-401.9068902509789 |24.44      |48.62   |0.082     |183.0                |189.4        |31238.80851 |19320.73171 |3    |17  |
|10767.6489 |16164.31858669042 |-5396.66968669042  |21.62      |73.2    |4.918     |0.073                |0.093        |21326.70366 |12594.29778 |8    |5   |
|16924.89028|18749.40723654926 |-1824.5169565492615|22.35      |75.1    |4.91      |92.5                 |75.1   

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual           |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|9825.542169|9774.205750195437 |51.336418804563436 |15.31      |87.0    |0.074     |0.062                |0.107        |22295.38462 |16452.89256 |11   |5   |
|26859.91632|26878.54486850245 |-18.62854850244912 |26.56      |69.8    |4.911     |695.6                |104.0        |35650.76412 |26643.03797 |7    |10  |
|8859.759036|10683.500280075641|-1823.7412440756416|19.45      |58.64   |0.081     |113.4                |81.9   

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual           |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|19115.73668|19438.69140306207 |-322.9547230620665 |24.33      |49.88   |4.902     |127.7                |89.5         |28966.21532 |21345.30095 |8    |7   |
|13714.28571|12968.677034390632|745.6086756093682  |25.37      |34.42   |4.918     |497.9                |68.8         |34654.52954 |21450.62241 |10   |15  |
|26833.45455|24146.231457039834|2687.2230929601646 |17.33      |80.7    |0.085     |0.04                 |0.093  

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual          |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|13775.42169|12670.089982324433|1105.3317076755666|12.34      |73.2    |0.076     |0.088                |0.152        |20931.64557 |13706.99088 |1    |3   |
|10925.76098|11066.574230096034|-140.8132500960346|20.49      |68.76   |0.298     |0.091                |0.107        |24926.0177  |15077.33888 |9    |5   |
|27259.18495|22557.460978873245|4701.723971126754 |26.42      |33.18   |4.905     |377.8                |317.4        

-------------------------------------------
Batch: 4
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual           |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|21657.6    |21878.227727521436|-220.62772752143792|22.88      |42.77   |0.08      |359.8                |367.2        |38806.88742 |23942.61954 |6    |18  |
|7610.60241 |9198.780774297946 |-1588.1783642979453|15.78      |63.63   |0.08      |84.3                 |21.27        |24116.92308 |19476.44628 |11   |8   |
|15020.71502|15235.06695936658 |-214.35193936658106|24.96      |59.49   |4.923     |565.7                |92.2   

...

-------------------------------------------
Batch: 15
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual           |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|9899.639856|11907.006157201518|-2007.3663012015186|14.76      |70.5    |0.084     |497.3                |36.39        |32285.93156 |26087.75698 |12   |12  |
|10181.9928 |11841.557368071757|-1659.564568071757 |14.04      |61.0    |0.082     |137.3                |136.8        |31884.41065 |23112.61123 |12   |11  |
|27728.65204|28994.208044018927|-1265.5560040189266|33.81      |33.38   |4.906     |218.1                |173.1 

-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual           |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|19147.95181|17229.88836721447 |1918.0634427855293 |14.44      |65.04   |0.082     |0.055                |0.096        |30167.08861 |20443.769   |1    |23  |
|28715.73668|28730.210375858525|-14.47369585852357 |23.88      |84.4    |4.922     |0.095                |0.085        |42410.47725 |27039.91552 |8    |22  |
|13848.51064|11530.850262545562|2317.6603774544383 |22.16      |61.2    |4.92      |516.2                |49.55 

-------------------------------------------
Batch: 17
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual           |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|10902.28091|13531.139178730944|-2628.8582687309445|15.02      |55.48   |0.086     |43.02                |43.42        |33143.72624 |27328.62841 |12   |16  |
|18297.17868|17175.411563937363|1121.7671160626378 |25.18      |88.4    |4.911     |76.7                 |57.12        |25571.58713 |17502.00634 |8    |7   |
|16438.06452|15504.481507979795|933.5830120202045  |15.77      |58.64   |4.92      |412.1                |429.0 

-------------------------------------------
Batch: 18
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual           |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|25931.56627|25272.996225242674|658.5700447573254  |10.29      |71.1    |0.079     |0.059                |0.1          |42185.31646 |27545.28875 |1    |21  |
|17303.13253|18733.890957857548|-1430.7584278575487|16.57      |51.05   |4.92      |516.7                |37.49        |34566.07595 |21151.36778 |1    |12  |
|18237.04615|18115.00744226004 |122.03870773995732 |23.62      |38.44   |0.082     |1050.0               |303.0 

-------------------------------------------
Batch: 19
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual           |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|16345.16129|17125.613911998375|-780.4526219983745 |13.13      |68.85   |0.077     |39.64                |36.69        |31753.53191 |17220.73171 |3    |11  |
|14625.54217|13185.451601923927|1440.0905680760734 |10.42      |81.3    |0.075     |0.073                |0.13         |21685.06329 |14250.45593 |1    |1   |
|9674.909964|11148.936441167873|-1474.0264771678721|11.41      |48.23   |0.076     |0.084                |0.089 

After the streaming finished processing all input files, I stopped the query.

In [19]:
# stop the query
query.stop()

# Overall Summary

In this project, the first part uses power‑consumption data from the UCI Machine Learning Repository to build an Elastic Net regression model with Spark MLlib. In the process, I incorporated SQL‑based feature engineering, applied one‑hot encoding, performed PCA, tuned hyperparameters through cross‑validation, and assembled everything into a complete transformation pipeline.

The second part extends this model into a streaming context. I wrote a Python script that continuously generates 20 CSV files to simulate an incoming data stream. Spark Structured Streaming then applies the fitted model to each micro‑batch, producing real‑time predictions, residuals, along with other predictors, printing the resulting data frames to the console.

Together, these two components demonstratehow a single Spark workflow can move smoothly from offline model training to live streaming inference, creating a coherent end‑to‑end pipeline.